In [2]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')


In [4]:
data1= r"C:\Users\rajeshkumar.t\Downloads\6025ac4bfe90ad7c55f5526d6e9af44a.csv"
df= pd.read_csv(data1)
print(df.columns)

Index(['oms_version', 'id', 'order_item_id', 'type', 'flow_type',
       'order_external_id', 'checkout_id', 'status', 'net_filter_flag', 'gmv',
       ...
       'brand_classification_key', 'cms_vertical', 'is_large', 'is_bulk_order',
       'order_sales_app', 'order_sales_experience', 'order_tags', 'unit_tags',
       'is_extra_saver_flag', 'order_date_key'],
      dtype='object', length=226)


In [5]:
buckets = df[['account_id','analytic_category']].drop_duplicates()
pairs = pd.merge(buckets, buckets, on = 'account_id', suffixes=('_A','_B'))
print(pairs.columns)

Index(['account_id', 'analytic_category_A', 'analytic_category_B'], dtype='object')


In [6]:
unique_pairs = pairs[pairs['analytic_category_A'] < pairs['analytic_category_B']]
mba_results = (unique_pairs.groupby(['analytic_category_A','analytic_category_B'])
               .size()
               .reset_index(name='Co_Occurrence_Count')
               .sort_values(by='Co_Occurrence_Count', ascending=False))
print("Top Product Affinities")
print(mba_results.head(10))

Top Product Affinities
    analytic_category_A               analytic_category_B  Co_Occurrence_Count
206             Gourmet  HouseHoldSuppliesAndConsummables                  519
202             Gourmet                          Grooming                  516
203             Gourmet                HealthAndNutrition                  158
256            Grooming  HouseHoldSuppliesAndConsummables                  151
168   FestiveAndGifting                           Gourmet                  100
241             Gourmet                    SpiritualItems                   70
255            Grooming                HealthAndNutrition                   46
15       BabyCareSupply                           Gourmet                   45
246             Gourmet                               Toy                   26
225             Gourmet            OverTheCounterMedicine                   24


In [7]:
#Support, Confidence, and Lift
total_orders = df['account_id'].nunique()
item_counts = df.groupby('analytic_category')['account_id'].nunique()
mba_results = mba_results.merge(item_counts.rename('count_A'),
                                left_on = 'analytic_category_A', right_index=True)
mba_results = mba_results.merge(item_counts.rename('count_B'),
                                left_on = 'analytic_category_B', right_index=True)

mba_results['Support'] = mba_results['Co_Occurrence_Count']/total_orders
mba_results['Confidence'] = mba_results['Co_Occurrence_Count'] /mba_results['count_A']
mba_results['Lift'] = (mba_results['Support'])/((mba_results['count_A']/total_orders) * (mba_results['count_B']/total_orders))

mba_results = mba_results[mba_results['Co_Occurrence_Count']>=2]
mba_results = mba_results[mba_results['Lift']>1.0]
mba_results = mba_results.sort_values(by='Lift',ascending=False)

print(mba_results[['analytic_category_A','analytic_category_B','Co_Occurrence_Count','Support','Confidence','Lift']].head(10))

    analytic_category_A analytic_category_B  Co_Occurrence_Count   Support  \
392   MensClothingJeans         MensTopwear                    3  0.000016   
106             Curtain      SpiritualItems                    2  0.000010   

     Confidence      Lift  
392    0.010101  1.637984  
106    0.006231  1.420271  


In [16]:
filtered_lift = mba_results[(mba_results['Co_Occurrence_Count'] >=3) & (mba_results['Lift']>1.0)]
print("After filtered Life value")
print(filtered_lift.sort_values(by='Lift', ascending=False)[['analytic_category_A','analytic_category_B','Co_Occurrence_Count','Lift']].head(5))

After filtered Life value
    analytic_category_A analytic_category_B  Co_Occurrence_Count      Lift
392   MensClothingJeans         MensTopwear                    3  1.637984
